# Advanced Real Estate Valuation & Predictive Modeling
**Big Data Analytics Course Project**

Dataset: King County House Sales (via DPhi GitHub CSV)

## 1. Problem Statement
Predict residential property prices using structural, geographic, and temporal features. Compare an interpretable Hedonic (linear) model against a non-linear XGBoost model, then extend with three novelties: Geohash micro-market clustering, Quantile risk bands, and a Residual-based arbitrage detector.

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor


## 3. Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/dphi-official/Datasets/master/kc_house_data/kc_house_data.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()


## 4. Exploratory Data Analysis

In [ ]:
df.info()
df.describe()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df['price'], bins=50)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Count")
plt.show()


## 5. Preprocessing & Feature Engineering

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['sale_year'] = df['date'].dt.year

# Derived temporal features
df['house_age'] = df['sale_year'] - df['yr_built']
df['was_renovated'] = (df['yr_renovated'] > 0).astype(int)
df['years_since_renovation'] = np.where(df['yr_renovated'] > 0,
                                         df['sale_year'] - df['yr_renovated'], 0)

# Log-transform skewed variables
df['log_price'] = np.log1p(df['price'])
df['log_sqft_living'] = np.log1p(df['sqft_living'])
df['log_sqft_lot'] = np.log1p(df['sqft_lot'])

df.drop(columns=['id', 'date', 'yr_renovated'], inplace=True)
df.head()


## 6. Train/Test Split

In [ ]:
features = ['bedrooms','bathrooms','log_sqft_living','log_sqft_lot','floors',
            'waterfront','view','condition','grade','sqft_above','sqft_basement',
            'yr_built','zipcode','lat','long','sqft_living15','sqft_lot15',
            'house_age','was_renovated','years_since_renovation']

X = df[features]
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 7. Model 1 — Hedonic Price Model (Linear Regression)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

hedonic_model = LinearRegression()
hedonic_model.fit(X_train_scaled, y_train)
hedonic_preds = hedonic_model.predict(X_test_scaled)

hedonic_rmse = np.sqrt(mean_squared_error(y_test, hedonic_preds))
hedonic_mae = mean_absolute_error(y_test, hedonic_preds)
hedonic_r2 = r2_score(y_test, hedonic_preds)

print(f"Hedonic Model -> RMSE: {hedonic_rmse:.4f}, MAE: {hedonic_mae:.4f}, R2: {hedonic_r2:.4f}")


## 8. Model 2 — XGBoost Regressor

In [ ]:
xgb_model = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_r2 = r2_score(y_test, xgb_preds)

print(f"XGBoost Model -> RMSE: {xgb_rmse:.4f}, MAE: {xgb_mae:.4f}, R2: {xgb_r2:.4f}")


## 9. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Hedonic (Linear)', 'XGBoost'],
    'RMSE': [hedonic_rmse, xgb_rmse],
    'MAE': [hedonic_mae, xgb_mae],
    'R2': [hedonic_r2, xgb_r2]
})
comparison


## 10. Novelty 1 — Dynamic Micro-Market Geohash Clustering
Converts latitude/longitude into geohash cells and injects a local median-price feature to capture street-level market variation that ZIP codes miss.

In [ ]:
import pygeohash as pgh

df['geohash'] = df.apply(lambda r: pgh.encode(r['lat'], r['long'], precision=6), axis=1)
geo_median = df.groupby('geohash')['price'].median().rename('micro_market_index')
df = df.merge(geo_median, on='geohash', how='left')

features_geo = features + ['micro_market_index']
X_geo = df[features_geo]

X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(X_geo, y, test_size=0.2, random_state=42)

xgb_geo = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_geo.fit(X_train_g, y_train_g)
geo_preds = xgb_geo.predict(X_test_g)

geo_rmse = np.sqrt(mean_squared_error(y_test_g, geo_preds))
geo_r2 = r2_score(y_test_g, geo_preds)
print(f"XGBoost + Geohash -> RMSE: {geo_rmse:.4f}, R2: {geo_r2:.4f}  (baseline XGBoost R2: {xgb_r2:.4f})")


## 11. Novelty 2 — Quantile Regression Risk Bands
Predicts the 10th, 50th, and 90th percentile prices to give a floor / expected / ceiling estimate instead of a single point value.

In [ ]:
quantiles = [0.1, 0.5, 0.9]
quantile_preds = {}

for q in quantiles:
    qmodel = XGBRegressor(objective='reg:quantileerror', quantile_alpha=q,
                           n_estimators=400, max_depth=6, learning_rate=0.05, random_state=42)
    qmodel.fit(X_train, y_train)
    quantile_preds[q] = qmodel.predict(X_test)

risk_bands = pd.DataFrame({
    'floor_10pct': np.expm1(quantile_preds[0.1]),
    'median_50pct': np.expm1(quantile_preds[0.5]),
    'ceiling_90pct': np.expm1(quantile_preds[0.9])
})
risk_bands.head()


## 12. Novelty 3 — Residual-Based Arbitrage / Hidden Gem Detector
Flags listings whose actual price falls well below the model's predicted fair value as potential underpriced opportunities.

In [ ]:
actual_price = np.expm1(y_test.values)
predicted_price = np.expm1(xgb_preds)

results = pd.DataFrame({'actual_price': actual_price, 'predicted_price': predicted_price})
results['residual'] = results['predicted_price'] - results['actual_price']
results['residual_pct'] = results['residual'] / results['predicted_price']

threshold = results['residual_pct'].quantile(0.90)  # top 10% most underpriced
hidden_gems = results[results['residual_pct'] >= threshold].sort_values('residual_pct', ascending=False)
hidden_gems.head(10)


## 13. Conclusion
- XGBoost outperforms the Hedonic linear model on RMSE/R², confirming non-linear feature interactions matter in real estate pricing.
- Geohash clustering adds incremental predictive power over ZIP-code-level granularity.
- Quantile risk bands provide investment-relevant uncertainty estimates beyond a single point prediction.
- The residual-based arbitrage detector translates prediction error into an actionable list of undervalued properties.